# Limpieza y unión del catálogo StreamView

Este notebook carga las fuentes oficiales, revisa su estructura, aplica limpieza verificable, homologa el esquema y exporta un catálogo integrado. No incluye EDA, gráficos ni análisis de negocio.

Regla de duplicados aplicada: se eliminan únicamente filas completamente idénticas. Los `show_id` repetidos entre Movies y TV Shows se conservan porque corresponden a contenidos distintos.

In [16]:
import pandas as pd

movies = pd.read_csv("/content/netflix_movies_detailed_up_to_2025.csv")
tv_shows = pd.read_csv("/content/netflix_tv_shows_detailed_up_to_2025.csv")

,filas,columnas,duplicados_completos,show_id_repetidos
Movies,16000,18,0,0
TV Shows,16000,16,0,9


In [21]:
movies.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471


In [22]:
tv_shows.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha...",South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members a...,1929.898,187,8.241
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by C...,1670.580,229,7.035
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooki...,1317.092,6,5.600
3,75685,TV Show,Prostřeno!,NaN,"Václav Vydra, Jana Boušková",Czech Republic,2010-03-01,2010,6.500,1 Seasons,Reality,cs,The knives (and forks) are out as a group of s...,1095.776,6,6.500
4,33847,TV Show,The Talk,NaN,"Amanda Kloots, Jerry O'Connell, Akbar Gbaja-Bi...","United States of America, Ireland",2010-10-18,2010,3.400,1 Seasons,Talk,en,A panel of well-known news and entertainment p...,712.070,12,3.400


In [23]:
revision = pd.DataFrame({
    'filas': [len(movies), len(tv_shows)],
    'columnas': [movies.shape[1], tv_shows.shape[1]],
    'duplicados_completos': [movies.duplicated().sum(), tv_shows.duplicated().sum()],
    'show_id_repetidos': [movies['show_id'].duplicated().sum(), tv_shows['show_id'].duplicated().sum()],
}, index=['Movies', 'TV Shows'])

display(revision)

,filas,columnas,duplicados_completos,show_id_repetidos
Movies,16000,18,0,0
TV Shows,16000,16,0,9


A traves de la siguiente visualización, podemos apreciar la distribución de nulos.

In [18]:
display(pd.DataFrame({
    'nulos_movies': movies.isna().sum(),
    'nulos_tv_shows': tv_shows.isna().sum(),
}))

,nulos_movies,nulos_tv_shows
budget,0,NaN
cast,204,1157.0
country,466,1797.0
date_added,0,0.0
description,132,3206.0
director,132,10965.0
duration,16000,0.0
genres,107,974.0
language,0,0.0
popularity,0,0.0




> Como observación, tanto en budget como en revenue, nulos_tv_shows los muestra como NaN, y esto es porque en realidad, aquellas columnas no existen en el dataset.



In [20]:
print('Columnas exclusivas de Movies:', sorted(set(movies.columns) - set(tv_shows.columns)))
print('Columnas exclusivas de TV Shows:', sorted(set(tv_shows.columns) - set(movies.columns)))

Columnas exclusivas de Movies: ['budget', 'revenue']
Columnas exclusivas de TV Shows: []


Esto nos muestra que movies solo tiene dos columnas extra, por lo que es posible una unión.

Se crea funcion capaz de transformar vacios a nulos.
Esto con el fin de estandarizar ambos datasets.


In [34]:
def normalize_text_columns(dataframe):
    cleaned = dataframe.copy()
    for column in cleaned.select_dtypes(include='object').columns:
        cleaned[column] = cleaned[column].str.strip().replace('', pd.NA)
    return cleaned

movies_clean = normalize_text_columns(movies)
tv_shows_clean = normalize_text_columns(tv_shows)

Para asegurar los datos, se sigue comprobando si existen duplicados
que puedan afectar al trabajo

In [38]:
# Solo se eliminan duplicados de filas completas, según DATA_RULES.md.
movies_clean = movies_clean.drop_duplicates().copy()
tv_shows_clean = tv_shows_clean.drop_duplicates().copy()

In [40]:
# Las variables exclusivas de Movies se mantienen
# como nulas para TV Shows.
for column in ['budget', 'revenue']:
    if column not in tv_shows_clean.columns:
        tv_shows_clean[column] = pd.NA

column_order = list(movies_clean.columns)
tv_shows_clean = tv_shows_clean.reindex(columns=column_order)

In [ ]:
for dataframe in [movies_clean, tv_shows_clean]:
    dataframe['show_id'] = dataframe['show_id'].astype('string')
    dataframe['date_added'] = pd.to_datetime(dataframe['date_added'], errors='raise')
    dataframe['release_year'] = pd.to_numeric(dataframe['release_year'], errors='raise').astype('Int64')
    dataframe['vote_count'] = pd.to_numeric(dataframe['vote_count'], errors='raise').astype('Int64')
    for column in ['rating', 'popularity', 'vote_average', 'budget', 'revenue']:
        dataframe[column] = pd.to_numeric(dataframe[column], errors='raise')

In [41]:
print(f'Movies: {len(movies)} -> {len(movies_clean)} filas')
print(f'TV Shows: {len(tv_shows)} -> {len(tv_shows_clean)} filas')

Movies: 16000 -> 16000 filas
TV Shows: 16000 -> 16000 filas


In [ ]:
catalogo = pd.concat([movies_clean, tv_shows_clean], ignore_index=True)

# Validaciones previas a la exportación.
assert set(catalogo['type'].dropna().unique()) == {'Movie', 'TV Show'}
assert not catalogo.duplicated().any(), 'Persisten filas completamente duplicadas.'
assert catalogo.loc[catalogo['type'].eq('TV Show'), ['budget', 'revenue']].isna().all().all()
assert raw_hashes_before == {
    'movies': file_hash(movies_path),
    'tv_shows': file_hash(tv_path),
}, 'Las fuentes originales fueron modificadas.'

catalogo['date_added'] = catalogo['date_added'].dt.strftime('%Y-%m-%d')
output_path.parent.mkdir(parents=True, exist_ok=True)
catalogo.to_csv(output_path, index=False)

print(f'Archivo exportado: {output_path}')
print(f'Filas: {len(catalogo):,}')
display(catalogo['type'].value_counts().rename_axis('type').to_frame('filas'))

Archivo exportado: /home/tomy/Downloads/Duoc/Visualizacion/visualizacion-de-datos-StreamView-Analytics/data/processed/catalogo_streamview.csv
Filas: 32,000


,filas
type,
Movie,16000
TV Show,16000


In [ ]:
catalogo_verificado = pd.read_csv(output_path)

assert len(catalogo_verificado) == len(catalogo)
assert set(catalogo_verificado['type'].unique()) == {'Movie', 'TV Show'}
assert not catalogo_verificado.duplicated().any()

print('Validación final completada correctamente.')
display(catalogo_verificado.info())

Validación final completada correctamente.
<class 'pandas.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       32000 non-null  int64  
 1   type          32000 non-null  str    
 2   title         32000 non-null  str    
 3   director      20903 non-null  str    
 4   cast          30639 non-null  str    
 5   country       29737 non-null  str    
 6   date_added    32000 non-null  str    
 7   release_year  32000 non-null  int64  
 8   rating        32000 non-null  float64
 9   duration      16000 non-null  str    
 10  genres        30919 non-null  str    
 11  language      32000 non-null  str    
 12  description   28660 non-null  str    
 13  popularity    32000 non-null  float64
 14  vote_count    32000 non-null  int64  
 15  vote_average  32000 non-null  float64
 16  budget        16000 non-null  float64
 17  revenue       16000 non-null  float64

None

Filtrar el DataFrame para solo incluir type == 'Movie' antes de calcular promedios o realizar análisis financieros que dependan de budget y revenue.
